# Day 077 — Exercise 4: capture_frames and analyze_stream

**What you'll build:** Full pipeline loops — capture and optionally analyze n frames from a camera with rate control.

**Why it matters:** These are the batch interfaces. `capture_frames` collects raw frames for offline processing. `analyze_stream` is the real-time vision loop — capture + analyze in one call.

In [ ]:
import numpy as np
from PIL import Image as _PILImage

def _make_mock_frame(h=100, w=100, val=50):
    return np.full((h, w, 3), val, dtype=np.uint8)

class _MockCap:
    def __init__(self, n=5, h=100, w=100):
        self._frames = [_make_mock_frame(h, w) for _ in range(n)]
        self._idx = 0
    def isOpened(self):
        return True
    def read(self):
        if self._idx >= len(self._frames):
            return False, None
        f = self._frames[self._idx]; self._idx += 1
        return True, f
    def release(self):
        pass
    def get(self, prop):
        return 0.0

_mock_camera_fn = lambda device: _MockCap(n=5)
_mock_analyze_fn = lambda img, q: 'FRAME:' + q[:12]
def open_camera(device=0, camera_fn=None):
    if camera_fn is not None:
        return camera_fn(device)
    import cv2
    cap = cv2.VideoCapture(device)
    if not cap.isOpened():
        raise RuntimeError(f'Cannot open camera device {device}')
    return cap

def read_frame(cap):
    return cap.read()
import io, base64

def frame_to_image(frame):
    from PIL import Image
    rgb = frame[:, :, ::-1]
    return Image.fromarray(rgb)

def analyze_frame(frame, question, analyze_fn=None):
    image = frame_to_image(frame)
    if analyze_fn is not None:
        return analyze_fn(image, question)
    import ollama
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    img_b64 = base64.b64encode(buf.getvalue()).decode()
    resp = ollama.chat(
        model='llava',
        messages=[{'role': 'user', 'content': question, 'images': [img_b64]}],
    )
    return resp['message']['content']
from pathlib import Path

def should_analyze(frame_count, every_n):
    return frame_count % every_n == 0

def save_frame(frame, path):
    out = Path(path)
    frame_to_image(frame).save(out, format='PNG')
    return out


## Task

1. `capture_frames(device=0, n_frames=5, every_n=1, camera_fn=None) -> list`
   - `cap = open_camera(device=device, camera_fn=camera_fn)`
   - `frames=[]; frame_count=0`; `try`: loop `while len(frames) < n_frames`:
     - `ret, frame = read_frame(cap)` → `break` if not ret
     - `if should_analyze(frame_count, every_n): frames.append(frame)`
     - `frame_count += 1`
   - `finally: cap.release()`; `return frames`

2. `analyze_stream(device=0, task=..., n_frames=5, every_n=1, camera_fn=None, analyze_fn=None) -> list[dict]`
   - Same loop but with `analyzed` counter (stops at `n_frames` analyzed)
   - Analyze passing frames: `analyze_frame(frame, task, analyze_fn=analyze_fn)`
   - Append `{'frame_idx': frame_count, 'description': description}`

## Your Implementation

In [ ]:
def capture_frames(device=0, n_frames=5, every_n=1, camera_fn=None):
    """Capture n_frames from a camera, rate-controlled by every_n."""
    raise NotImplementedError

def analyze_stream(device=0, task='Describe what you see.', n_frames=5,
                   every_n=1, camera_fn=None, analyze_fn=None):
    """Capture and analyze n_frames. Returns list of {frame_idx, description}."""
    raise NotImplementedError


In [ ]:
def capture_frames(device=0, n_frames=5, every_n=1, camera_fn=None):
    cap = open_camera(device=device, camera_fn=camera_fn)
    frames = []
    frame_count = 0
    try:
        while len(frames) < n_frames:
            ret, frame = read_frame(cap)
            if not ret:
                break
            if should_analyze(frame_count, every_n):
                frames.append(frame)
            frame_count += 1
    finally:
        cap.release()
    return frames

def analyze_stream(device=0, task='Describe what you see.', n_frames=5,
                   every_n=1, camera_fn=None, analyze_fn=None):
    cap = open_camera(device=device, camera_fn=camera_fn)
    results = []
    frame_count = 0
    analyzed = 0
    try:
        while analyzed < n_frames:
            ret, frame = read_frame(cap)
            if not ret:
                break
            if should_analyze(frame_count, every_n):
                description = analyze_frame(frame, task, analyze_fn=analyze_fn)
                results.append({'frame_idx': frame_count, 'description': description})
                analyzed += 1
            frame_count += 1
    finally:
        cap.release()
    return results


## Automated checks

In [ ]:

score, total = 0, 5
try:
    import numpy as np

    frames = capture_frames(n_frames=3, camera_fn=_mock_camera_fn)
    assert isinstance(frames, list) and len(frames) == 3
    score += 1; print("✅ capture_frames returns 3 frames")

    assert all(isinstance(f, np.ndarray) for f in frames)
    score += 1; print("✅ each frame is an ndarray")

    # every_n=2: frames 0, 2, 4 -> 3 frames from a 6-frame cap
    cap6 = _MockCap(n=6)
    frames2 = capture_frames(n_frames=3, every_n=2,
                              camera_fn=lambda d: _MockCap(n=6))
    assert len(frames2) == 3
    score += 1; print("✅ capture_frames respects every_n")

    results = analyze_stream(n_frames=2, camera_fn=_mock_camera_fn,
                              analyze_fn=_mock_analyze_fn)
    assert isinstance(results, list) and len(results) == 2
    score += 1; print("✅ analyze_stream returns 2 result dicts")

    assert all('frame_idx' in r and 'description' in r for r in results)
    score += 1; print("✅ each result has frame_idx and description")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def capture_frames(device=0, n_frames=5, every_n=1, camera_fn=None):
    cap = open_camera(device=device, camera_fn=camera_fn)
    frames = []
    frame_count = 0
    try:
        while len(frames) < n_frames:
            ret, frame = read_frame(cap)
            if not ret:
                break
            if should_analyze(frame_count, every_n):
                frames.append(frame)
            frame_count += 1
    finally:
        cap.release()
    return frames

def analyze_stream(device=0, task='Describe what you see.', n_frames=5,
                   every_n=1, camera_fn=None, analyze_fn=None):
    cap = open_camera(device=device, camera_fn=camera_fn)
    results = []
    frame_count = 0
    analyzed = 0
    try:
        while analyzed < n_frames:
            ret, frame = read_frame(cap)
            if not ret:
                break
            if should_analyze(frame_count, every_n):
                description = analyze_frame(frame, task, analyze_fn=analyze_fn)
                results.append({'frame_idx': frame_count, 'description': description})
                analyzed += 1
            frame_count += 1
    finally:
        cap.release()
    return results
```

**Why two counters in analyze_stream?** `frame_count` tracks all frames read (drives the every_n modulo check). `analyzed` tracks frames sent to the LLM (drives the stopping condition). Mixing them would break rate control.

</details>